Required Imports

In [ ]:

!pip install torch transformers accelerate  tenacity

!!pip install torch --extra-index-url https://download.pytorch.org/whl/cu118

!pip install sentence-transformers
!pip install evaluate rouge_score nltk

In [ ]:
!pip uninstall torch torchvision torchaudio transformers -y
!pip install --upgrade pip setuptools wheel

Found existing installation: torch 2.0.1+cu118
Uninstalling torch-2.0.1+cu118:
  Successfully uninstalled torch-2.0.1+cu118
Found existing installation: torchvision 0.15.2+cu118
Uninstalling torchvision-0.15.2+cu118:
  Successfully uninstalled torchvision-0.15.2+cu118
Found existing installation: torchaudio 2.0.2+cu118
Uninstalling torchaudio-2.0.2+cu118:
  Successfully uninstalled torchaudio-2.0.2+cu118
Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-2

In [ ]:

!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.36.2 sentencepiece protobuf accelerate datasets

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 126.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 66.1 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 2.0.0
    Uninstalling triton-2.0.0:
      Successfully uninstalled triton-2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [torchaudio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.15.2 requires transformers, which is not installed.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 129.9 MB/s eta 0:00:00
   ━━━━

Finetune Lora

In [ ]:

import os
import time
import json
import torch
import sys
from packaging import version


try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
        TrainingArguments,
        EarlyStoppingCallback,
    )
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
    from trl import SFTTrainer
    from datasets import Dataset
    from sklearn.model_selection import train_test_split
    import accelerate
    import transformers
    import peft
except ImportError as e:
    print(" Critical imports failed! Please install:")
    print("!pip install transformers==4.41.0 peft==0.10.0 accelerate==0.27.2 bitsandbytes==0.42.0 trl==0.8.6 scikit-learn")
    raise

# Verify versions
required_versions = {
    'transformers': '4.41.0',
    'peft': '0.10.0',
    'accelerate': '0.27.2'
}

for pkg, ver in required_versions.items():
    current_version = globals()[pkg].__version__
    if version.parse(current_version) != version.parse(ver):
        print(f" Version mismatch: {pkg}=={current_version} (required {ver})")
        sys.exit(1)

# Check GPU compatibility
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This script requires GPU acceleration.")
if torch.cuda.get_device_capability()[0] < 8:  # Ampere architecture check
    print(" Warning: Your GPU may not fully support bfloat16 operations")


MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_PATH = "unani_train_en_ur.json"
OUTPUT_DIR = f"unani-lora-ft-{int(time.time())}"
HF_TOKEN = ""  # Replace with your token

os.makedirs(OUTPUT_DIR, exist_ok=True)


def format_instruction(sample):
    system_prompt = """You are an expert in Unani medicine. Provide accurate, detailed information about herbs, treatments, and remedies in both English and Urdu when available."""


    instruction = sample.get('instruction', '')
    output = sample.get('output', '')
    herb = sample.get('herb', 'an herb')

    # Get Urdu fields if available
    urdu_instruction = sample.get('instruction_ur', '')
    urdu_output = sample.get('output_ur', '')

    # Format the prompt
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{instruction}
{urdu_instruction if urdu_instruction else ''}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{output}
{urdu_output if urdu_output else ''}<|eot_id|>"""

def load_and_process_data(file_path, test_size=0.1):
    with open(file_path) as f:
        data = json.load(f)

    # Validate data structure
    required_fields = ['instruction', 'output']
    valid_data = []

    for item in data:
        if not isinstance(item, dict):
            print(f" Skipping non-dict item: {item}")
            continue

        missing_fields = [field for field in required_fields if field not in item]
        if missing_fields:
            print(f" Skipping item missing fields {missing_fields}: {item.get('herb', 'Unknown')}")
            continue

        valid_data.append(item)

    if not valid_data:
        raise ValueError(" No valid data found after filtering")

    print(f" Loaded {len(valid_data)} valid samples")

    # Format all valid samples
    formatted_data = [{"text": format_instruction(sample)} for sample in valid_data]

    # Split into train and validation
    train_data, val_data = train_test_split(formatted_data, test_size=test_size, random_state=42)

    return train_data, val_data


def load_model():

    for quant_config in [
        {  # 4-bit
            "load_in_4bit": True,
            "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_compute_dtype": torch.float16
        },
        {  # 8-bit
            "load_in_8bit": True,
            "bnb_8bit_compute_dtype": torch.float16
        },
        None  # No quantization
    ]:
        try:
            print(f"Attempting config: {quant_config}")
            bnb_config = BitsAndBytesConfig(**quant_config) if quant_config else None

            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                quantization_config=bnb_config,
                device_map="auto",
                token=HF_TOKEN,
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True
            )
            print(f" Success with config: {quant_config}")
            return model
        except Exception as e:
            print(f" Failed: {str(e)}")
            torch.cuda.empty_cache()
    raise RuntimeError(" All quantization attempts failed")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    padding_side="right",
    use_fast=False,
    truncation_side="left"
)
tokenizer.pad_token = tokenizer.eos_token


model = load_model()


try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    print(" Prepared model for k-bit training")
except Exception as e:
    print(f" Couldn't prepare for k-bit training: {str(e)}")
    print("Proceeding without k-bit optimization")

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
print(" Configured PEFT model")

# Data preparation
try:
    train_data, val_data = load_and_process_data(DATASET_PATH)
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)
    print(f" Loaded {len(train_data)} training and {len(val_data)} validation samples")
except Exception as e:
    print(f" Failed to load data: {str(e)}")
    raise


if torch.cuda.is_bf16_supported():
    print(" BF16 supported - using mixed precision training")
    mixed_precision = "bf16"
else:
    print(" BF16 not supported - falling back to FP16")
    mixed_precision = "fp16"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=(mixed_precision == "fp16"),
    bf16=(mixed_precision == "bf16"),
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    warmup_ratio=0.1,
    max_grad_norm=0.3,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    load_best_model_at_end=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


try:
    print(" Starting training...")
    trainer.train()

    # Save outputs
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    # Save training stats
    with open(os.path.join(OUTPUT_DIR, "training_stats.json"), "w") as f:
        json.dump(trainer.state.log_history, f)

    print(f" Training complete! Model saved to: {OUTPUT_DIR}")

except Exception as e:
    print(f" Training failed: {str(e)}")
    raise
finally:
    torch.cuda.empty_cache()

Overwriting finetune_lora.py


In [ ]:
!python finetune_lora.py

2025-07-26 19:34:32.947906: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753558472.975949    6424 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753558472.985032    6424 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-26 19:34:33.019794: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
⚠️ Warning: Your GPU may not fully support bfloat16 operations
Attempting config: {'load_in_4bit': True, 'bnb_4bit_quant_type